In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from os import walk
import pickle
from pathlib import Path

# Set project root
project_root = Path().resolve().parents[0]
import geopandas as gpd
from geopandas import GeoDataFrame
from shapely.geometry import Point


In [ ]:
def find_nearest(value,vector):
    vector = np.float64(vector)-np.float64(value);
    return np.argmin(np.abs(vector))

In [ ]:
#restrepo_features_path = project_root / "data" / "raw" / "features" / "FeaturesPhrampusLee_TOCSedRate_updated"
#restrepo_features_path = Path().resolve().parents[3] / "work_old/Data/Features_PhrampusLee_noNaN_highres_rot/"
restrepo_features_path = project_root / "data" / "raw" / "features" / "FeaturesPhrampusLee_TOCSedRate_updated"
restrepo_files = []

for (dirpath, dirnames, filenames) in walk(restrepo_features_path):
    #print(filenames)
    restrepo_files.extend(filenames)

restrepo_files.sort()
len(restrepo_files)



In [ ]:
feature_selection = pd.read_csv(project_root / "data" / "interim" / "selectedFeatureLists" / "selectedfeatures_v_men.txt").values
#feature_selection = pd.read_csv(Path("restrepoHighResFeaturesRaw-50.txt")).values


selection_filter = np.zeros(len(restrepo_files), dtype=bool)
for idx, feature_name in enumerate(restrepo_files):
    if feature_name in feature_selection: # drop '.nc' from feature names #drop .grd
        selection_filter[idx] = True

In [ ]:
feature_selection.shape

In [ ]:
filenames

In [ ]:
selected_files = []
for file in restrepo_files:
    #if file in feature_selection:
        selected_files.append(file)

In [ ]:
len(feature_selection)

In [ ]:
restrepo_files = selected_files

In [ ]:



restrepo_files

In [ ]:
#"""
file_idx = -1
corrupted_files = 0

for file in restrepo_files[0:len(restrepo_files)]:
    file_idx += 1
    
    try:
        rootgrp = xr.open_dataset(Path(restrepo_features_path / file), format = 'NETCDF4')
        
        if file == "SF_POROSITY_FRC.5m_with_coords_flip.nc":
            porosity_fileidx = file_idx
            print("porosity_fileidx_lee")
            print(porosity_fileidx)
            dataset_numpy_porosity = rootgrp.variables['data']

        elif file == "POROSITY_global_prediction.grd":
            porosity_fileidx_martin = file_idx
            print("porosity_fileidx_martin") 
            print(porosity_fileidx_martin)
            dataset_numpy_porosity_martin = rootgrp.variables['data']

            
        elif file == "SEAFLOOR_DENSITY_GCM3.nc":
            density_fileidx = file_idx
            print("density_fileidx")
            print(density_fileidx)
            dataset_numpy_density = rootgrp.variables['data']
            

    except:
        #restrepo_X_mean = np.append(restrepo_X_mean,np.nan)
        #restrepo_X_std = np.append(restrepo_X_std,np.nan)
        corrupted_files += 1
        continue
#"""

In [ ]:
"""
import matplotlib.pyplot as plt
import os

output_dir = Path().resolve().parents[3] / "work_old/Data/Features_Images"
os.makedirs(output_dir, exist_ok=True)

file_idx = -1
corrupted_files = 0

for file in restrepo_files[0:len(restrepo_files)]:
    file_idx += 1
    
    try:
        rootgrp = xr.open_dataset(Path(restrepo_features_path / file), format = 'NETCDF4')
        print(f"\nFile: {file}")
        
        # Plot the data
        plt.figure(figsize=(10,8))
        plt.imshow(rootgrp.variables['data'], 
                  extent=[rootgrp.lon.values[0], rootgrp.lon.values[-1],
                         rootgrp._LOG_MG_M3_lat.values[0], rootgrp.lat.values[-1]],
                  aspect='auto')
        plt.colorbar(label='Value')
        plt.title(file)
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        
        # Save plot
        output_file = output_dir / f"{file.split('.')[0]}.png"
        plt.savefig(output_file)
        plt.close()

    except:
        restrepo_X_mean = np.append(restrepo_X_mean,np.nan)
        restrepo_X_std = np.append(restrepo_X_std,np.nan)
        corrupted_files += 1
        continue
"""

In [ ]:

land_file = "../../data/raw/island_map.npy"
land_map = np.load(land_file)
land_map[np.isnan(land_map)] = 1

# Calculate porosity from bulk density using Graw et al. (2021) equation
dS = 2.65  # density of dry sediments (g/cm3)
dPW = 1.024  # density of porewater (g/cm3)
dataset_numpy_porosity_graw = (dS - dataset_numpy_density) / (dS - dPW)

dataset_numpy_porosity_graw = dataset_numpy_porosity_graw*100.0

# Calculate dry bulk density (DBD)
DBD = (1 - dataset_numpy_porosity_graw/100.0) * dS

# Create masked arrays and rotate/flip
porosity_martin = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity_martin)))
porosity_lee = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity)))
density = np.rot90(np.rot90(np.fliplr(dataset_numpy_density)))
porosity_graw = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity_graw)))
DBD = np.rot90(np.rot90(np.fliplr(DBD)))
#land_map_rotated = np.rot90(np.rot90(np.fliplr(land_map)))

porosity_martin_masked = np.ma.masked_array(porosity_martin, mask=land_map)
porosity_lee_masked = np.ma.masked_array(porosity_lee, mask=land_map)
density_masked = np.ma.masked_array(density, mask=land_map)
porosity_graw_masked = np.ma.masked_array(porosity_graw, mask=land_map)
DBD_masked = np.ma.masked_array(DBD, mask=land_map)

fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(1, 5, figsize=(25, 3))

im1 = ax1.imshow(porosity_martin_masked, vmin=40, vmax=100, cmap='cubehelix')
fig.colorbar(im1, ax=ax1)
ax1.set_title("Porosity (Martin et al., 2015)")

im2 = ax2.imshow(porosity_lee_masked, vmin=40, vmax=100, cmap='cubehelix')
fig.colorbar(im2, ax=ax2)
ax2.set_title("Porosity (Lee et al., 2019)")



# Plot wet density (original), porosity, and dry bulk density
im3 = ax3.imshow(density_masked, vmin=1.2, vmax=1.8, cmap='cubehelix')
fig.colorbar(im3, ax=ax3)
ax3.set_title("Wet Bulk Density (Graw et al., 2021)")

# Add a fourth subplot for porosity
#ax4 = fig.add_subplot(1, 4, 4)
im4 = ax4.imshow(porosity_graw_masked, vmin=40, vmax=100, cmap='cubehelix')
fig.colorbar(im4, ax=ax4)
ax4.set_title("Porosity (Graw-derived)")

# Add a fifth subplot for DBD
#ax5 = fig.add_subplot(1, 5, 5)
im5 = ax5.imshow(DBD_masked, vmin=0.6, vmax=1.6, cmap='cubehelix')
fig.colorbar(im5, ax=ax5)
ax5.set_title("Dry Bulk Density (DBD)")

plt.tight_layout()


In [ ]:

# Load land mask
land_file = "../../data/raw/island_map.npy"
land_map = np.load(land_file)
land_map[np.isnan(land_map)] = 1

# Calculate porosity from bulk density using Graw et al. (2021) equation
dS = 2.65  # density of dry sediments (g/cm3)
dPW = 1.024  # density of porewater (g/cm3)
dataset_numpy_porosity_graw = (dS - dataset_numpy_density) / (dS - dPW)
dataset_numpy_porosity_graw = dataset_numpy_porosity_graw * 100.0

# Prepare porosity maps (rotate/flip and mask)
porosity_martin = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity_martin)))
porosity_lee = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity)))
porosity_graw = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity_graw)))

porosity_martin_masked = np.ma.masked_array(porosity_martin, mask=land_map)
porosity_lee_masked = np.ma.masked_array(porosity_lee, mask=land_map)
porosity_graw_masked = np.ma.masked_array(porosity_graw, mask=land_map)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

im1 = ax1.imshow(porosity_martin_masked, vmin=40, vmax=100, cmap='cubehelix')
fig.colorbar(im1, ax=ax1)
ax1.set_title("Porosity (Martin et al., 2015)")

im2 = ax2.imshow(porosity_lee_masked, vmin=40, vmax=100, cmap='cubehelix')
fig.colorbar(im2, ax=ax2)
ax2.set_title("Porosity (Lee et al., 2019)")

im3 = ax3.imshow(porosity_graw_masked, vmin=40, vmax=100, cmap='cubehelix')
fig.colorbar(im3, ax=ax3)
ax3.set_title("Porosity (Graw-derived)")

# Highlight points where porosity < 55 for each map and calculate percentage
percentages = []
for ax, porosity_map, mask_map, title in zip(
    [ax1, ax2, ax3],
    [porosity_martin, porosity_lee, porosity_graw],
    [porosity_martin_masked.mask, porosity_lee_masked.mask, porosity_graw_masked.mask],
    ["Martin et al., 2015", "Lee et al., 2019", "Graw-derived"]
):
    # Find indices where porosity < 55 and not masked
    valid_mask = ~mask_map
    low_porosity_mask = (porosity_map < 55) & valid_mask
    y_idx, x_idx = np.where(low_porosity_mask)
    ax.scatter(x_idx, y_idx, s=10, c='red', marker='o', label='Porosity < 55')
    ax.legend(loc='lower right')
    # Calculate percentage
    n_total = np.sum(valid_mask)
    n_low = np.sum(low_porosity_mask)
    percent = 100.0 * n_low / n_total if n_total > 0 else np.nan
    percentages.append(percent)
    # Annotate percentage on the plot
    ax.text(0.02, 0.98, f"{percent:.2f}% < 55", color='red', fontsize=12,
            ha='left', va='top', transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

plt.tight_layout()

# --- NEW CODE: Check how many MAR points are in low porosity areas ---

# Assume you have two dataframes: df_1 and df_2, each with columns:
# ['Latitude', 'Longitude', 'MAR(g/cm**2/ka)', 'Sedimentation rate [cm/ka]']
# (If not, adjust the dataframe names accordingly.)

import copy
df_1 = pd.read_csv(str(Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/labels/sedrate_continentalshelves.csv"))
df_2 = pd.read_csv(str(Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/labels/sedrate_deep.csv"))
def count_low_porosity_points(df, porosity_map, mask_map, threshold=55):
    """
    For each point in df (with Latitude, Longitude), check if it falls in a low porosity (<threshold) area.
    Returns the count and a boolean mask for the dataframe.
    """
    # The porosity maps are on a regular grid: assume global 360x180 (1 deg), or get shape
    n_lat, n_lon = porosity_map.shape
    #print(n_lat, n_lon)
    # Build grid axes
    lon_grid = np.linspace(-180, 180, n_lon)
    lat_grid = np.linspace(-90, 90, n_lat)
    # For each point, find nearest grid cell
    low_mask = []
    for idx, row in df.iterrows():
        lat = row['Latitude']
        lon = row['Longitude']
        # Find nearest index
        lat_idx = np.abs(lat_grid - lat).argmin()
        lon_idx = np.abs(lon_grid - lon).argmin()
        # Check if masked (land) or not
        if mask_map[lat_idx, lon_idx]:
            low_mask.append(False)
            continue
        # Check porosity value
        if porosity_map[lat_idx, lon_idx] < threshold:
            low_mask.append(True)
        else:
            low_mask.append(False)
    n_low = np.sum(low_mask)
    return n_low, np.array(low_mask)

# Example for df_1 and porosity_graw
n_low_graw_df1, mask_low_graw_df1 = count_low_porosity_points(df_1, porosity_graw, porosity_graw_masked.mask)
n_low_lee_df1, mask_low_lee_df1 = count_low_porosity_points(df_1, porosity_lee, porosity_lee_masked.mask)
n_low_martin_df1, mask_low_martin_df1 = count_low_porosity_points(df_1, porosity_martin, porosity_martin_masked.mask)

print(f"df_1: {n_low_graw_df1} points in low porosity (<55) areas (Graw-derived)")
print(f"df_1: {n_low_lee_df1} points in low porosity (<55) areas (Lee et al., 2019)")
print(f"df_1: {n_low_martin_df1} points in low porosity (<55) areas (Martin et al., 2015)")

# If you have a second dataframe, repeat:
if 'df_2' in locals():
    n_low_graw_df2, mask_low_graw_df2 = count_low_porosity_points(df_2, porosity_graw, porosity_graw_masked.mask)
    n_low_lee_df2, mask_low_lee_df2 = count_low_porosity_points(df_2, porosity_lee, porosity_lee_masked.mask)
    n_low_martin_df2, mask_low_martin_df2 = count_low_porosity_points(df_2, porosity_martin, porosity_martin_masked.mask)
    print(f"df_2: {n_low_graw_df2} points in low porosity (<55) areas (Graw-derived)")
    print(f"df_2: {n_low_lee_df2} points in low porosity (<55) areas (Lee et al., 2019)")
    print(f"df_2: {n_low_martin_df2} points in low porosity (<55) areas (Martin et al., 2015)")

# Optionally, add a column to the dataframe to mark low porosity points
df_1 = copy.deepcopy(df_1)
df_1['LowPorosity_Graw'] = mask_low_graw_df1
df_1['LowPorosity_Lee'] = mask_low_lee_df1
df_1['LowPorosity_Martin'] = mask_low_martin_df1

if 'df_2' in locals():
    df_2 = copy.deepcopy(df_2)
    df_2['LowPorosity_Graw'] = mask_low_graw_df2
    df_2['LowPorosity_Lee'] = mask_low_lee_df2
    df_2['LowPorosity_Martin'] = mask_low_martin_df2
n_low_graw_df1, mask_low_graw_df1 = count_low_porosity_points(df_1, porosity_graw, porosity_graw_masked.mask)

In [ ]:
#check if the points are being taken properly form the porosity map
"""
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 8))
plt.imshow(dataset_numpy_porosity, 
           extent=[-180, 180, -90, 90], 
           origin='lower', 
           cmap='viridis', 
           aspect='auto')

plt.colorbar(label='Porosity (%)')
plt.title("Porosity Map with MAR Locations")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.scatter(df_1["Longitude"], df_1["Latitude"], 
            c='red', s=10, label="Sedimentation Data", alpha=0.6)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
"""

In [ ]:
def convertSR2MAR(sedrate, porosity, ds):
    """Convert sedimentation rate (cm/ka) to MAR (g/cm²/ka)."""
    MAR = (1 - porosity / 100.0) * ds * (sedrate )
    return MAR
def convertMAR2SR(MAR, porosity, ds):
    """Convert MAR (g/cm²/ka) back to sedimentation rate (cm/ka)."""
    if porosity >= 100:  # avoid division by zero or negative values
        return np.nan
    return MAR / ((1 - porosity / 100.0) * ds)


### df_1

In [ ]:
data_path = Path("../../data/raw/labels/") 
sedrate_labels_file = "sedRateData_mosaic_pangea.csv"


In [ ]:
df_1 = pd.read_csv(Path(data_path / sedrate_labels_file))

In [ ]:
#df_1 = df_1[df_1["Depth [m]"]<=0.01]
df_1

In [ ]:
df_1.columns

In [ ]:
def fix_sampledepth_and_calculate_average(df, sampledepth_col='Sampledepth [cm]'):
    """
    Fix sampledepth column by converting date-like entries back to ranges
    and calculate average depth for ranges.
    
    Args:
        df: DataFrame containing the sampledepth column
        sampledepth_col: Name of the sampledepth column
    
    Returns:
        DataFrame with fixed sampledepth and new average depth column
    """
    
    # Month abbreviation to number mapping
    month_map = {
        'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06',
        'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
    }
    
    def fix_sampledepth(val):
        """Convert date-like entries back to range format"""
        if pd.isna(val):
            return val
        
        if isinstance(val, str):
            # Match patterns like '1-Mar', '4-Jun', etc.
            match = re.match(r'^(\d{1,2})-([A-Za-z]{3})$', val)
            if match:
                start = int(match.group(1))
                month_abbr = match.group(2).title()
                end = month_map.get(month_abbr)
                if end:
                    return f"{start:02d}-{end}"
            
            # Also handle cases like '0-4', '9-Oct' (where second part is month)
            match2 = re.match(r'^(\d{1,2})-([A-Za-z]{3})$', val)
            if match2:
                start = int(match2.group(1))
                month_abbr = match2.group(2).title()
                end = month_map.get(month_abbr)
                if end:
                    return f"{start:02d}-{end}"
        
        return val
    
    def calculate_average_depth(val):
        """Calculate average depth for range values"""
        if pd.isna(val):
            return np.nan
        
        if isinstance(val, str):
            # Check if it's a range (contains '-')
            if '-' in val:
                try:
                    parts = val.split('-')
                    if len(parts) == 2:
                        start = float(parts[0])
                        end = float(parts[1])
                        return (start + end) / 2
                except (ValueError, TypeError):
                    return np.nan
            else:
                # Single value
                try:
                    return float(val)
                except (ValueError, TypeError):
                    return np.nan
        
        elif isinstance(val, (int, float)):
            return float(val)
        
        return np.nan
    
    # Create a copy to avoid modifying original
    df_fixed = df.copy()
    
    # Fix the sampledepth column
    df_fixed[sampledepth_col] = df_fixed[sampledepth_col].apply(fix_sampledepth)
    
    # Create new column with average depth
    avg_col_name = 'Sampledepth_avg_cm'
    df_fixed[avg_col_name] = df_fixed[sampledepth_col].apply(calculate_average_depth)
    
    return df_fixed


In [ ]:
df_1 = df_1[["Latitude","Longitude","Sedimentation rate [cm/ka]"]]
df_1 = df_1[df_1["Longitude"]>-180]
df_1 = df_1.dropna(how="any")

In [ ]:
df_1

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_1 = df_1.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_1)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_1 = df_1.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_1)

In [ ]:
import numpy as np

# Preprocessing (already done)
df_1 = df_1[["Latitude", "Longitude", "Sedimentation rate [cm/ka]"]]
df_1 = df_1[df_1["Longitude"] > -180]
df_1 = df_1.dropna(how="any")

# Assuming dataset_numpy_porosity, latitudes, longitudes, ds are defined
#dataset_numpy_porosity = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity)))


"""
def convertSR2MARusingDensity(sedrate, density):
    #Convert sedimentation rate (cm/ka) to MAR (g/cm²/ka) using density.
    MAR = density * (sedrate / 1000.0)
    return MAR
"""
# Constant 
ds = 2.6  # dry sediment density in g/cm³ 

# Compute MAR for each row using both methods
MAR_list = []
MAR_density_list = []
MAR_graw_list = []
MAR_martin_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_1.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity[lat_idx, lon_idx])
    porosity_martin = int(dataset_numpy_porosity_martin[lat_idx, lon_idx])
    porosity_graw = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])

    # Calculate MAR using porosity method
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

    MAR_martin = convertSR2MAR(sedrate, porosity_martin, ds)
    MAR_martin_list.append(MAR_martin)

    MAR_graw = convertSR2MAR(sedrate, porosity_graw, ds)
    MAR_graw_list.append(MAR_graw)
    """
    # Calculate MAR using density method
    density = int(dataset_numpy_density[lat_idx, lon_idx])  # Calculate density from porosity and ds
    MAR_density = convertSR2MARusingDensity(sedrate, density)
    MAR_density_list.append(MAR_density)
    """

# Add both MAR columns to the DataFrame
#df_1["MAR_Lee_porosity (g/cm**2/ka)"] = MAR_list
#df_1["MAR_density (g/cm**2/yr)"] = MAR_density_list
#df_1["MAR_martin (g/cm**2/ka)"] = MAR_martin_list
df_1["MAR(g/cm**2/ka)"] = MAR_graw_list






In [ ]:
df_1

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_1['Longitude'], df_1['Latitude'])]
gdf_1 = GeoDataFrame(df_1, geometry=geometry)  
gdf_1.plot(ax = world.plot(figsize = (20, 12)), column = df_1["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_1['Longitude'], df_1['Latitude'])]
gdf_1 = GeoDataFrame(df_1, geometry=geometry)  
gdf_1.plot(ax = world.plot(figsize = (20, 12)), column = df_1["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_1.describe()

In [ ]:
# Plot 5 regions as a tile (1 row, 5 columns) for sedimentation rate and for MAR

import matplotlib.pyplot as plt

# Define region boundaries (example: Atlantic, Pacific, Indian, Arctic, Southern)
regions = {
    "Atlantic":    {"lat_min": -60, "lat_max": 70, "lon_min": -70, "lon_max": 20},
    "Pacific":     {"lat_min": -60, "lat_max": 70, "lon_min": 120, "lon_max": -70},  # Note: Pacific crosses dateline
    "Indian":      {"lat_min": -60, "lat_max": 30, "lon_min": 20, "lon_max": 120},
    "Arctic":      {"lat_min": 70,  "lat_max": 90, "lon_min": -180, "lon_max": 180},
    "Southern":    {"lat_min": -90, "lat_max": -60, "lon_min": -180, "lon_max": 180},
}

def region_mask(df, region):
    lat_mask = (df['Latitude'] >= region['lat_min']) & (df['Latitude'] <= region['lat_max'])
    # Handle Pacific region crossing the dateline
    if region['lon_min'] < region['lon_max']:
        lon_mask = (df['Longitude'] >= region['lon_min']) & (df['Longitude'] <= region['lon_max'])
    else:
        lon_mask = (df['Longitude'] >= region['lon_min']) | (df['Longitude'] <= region['lon_max'])
    return lat_mask & lon_mask

region_names = list(regions.keys())

# --- Sedimentation rate tile plot ---
fig, axes = plt.subplots(1, 5, figsize=(28, 6), constrained_layout=True)
for i, region_name in enumerate(region_names):
    bounds = regions[region_name]
    df_region = df_1[region_mask(df_1, bounds)]
    ax = axes[i]
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world.plot(ax=ax, color='lightgray', edgecolor='black')
    if not df_region.empty:
        geometry = [Point(xy) for xy in zip(df_region['Longitude'], df_region['Latitude'])]
        gdf_region = GeoDataFrame(df_region, geometry=geometry)
        gdf_region.plot(
            ax=ax, 
            column="Sedimentation rate [cm/ka]", 
            vmin=0, vmax=5000, 
            legend=(i==4),  # Only show legend on last plot
            cmap='plasma', 
            markersize=40
        )
    ax.set_title(f"{region_name}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    if i != 4:
        ax.get_legend().remove() if ax.get_legend() else None
fig.suptitle("Sedimentation rate [cm/ka] by Region", fontsize=18)
plt.show()

# --- MAR tile plot ---
fig, axes = plt.subplots(1, 5, figsize=(28, 6), constrained_layout=True)
for i, region_name in enumerate(region_names):
    bounds = regions[region_name]
    df_region = df_1[region_mask(df_1, bounds)]
    ax = axes[i]
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world.plot(ax=ax, color='lightgray', edgecolor='black')
    if not df_region.empty:
        geometry = [Point(xy) for xy in zip(df_region['Longitude'], df_region['Latitude'])]
        gdf_region = GeoDataFrame(df_region, geometry=geometry)
        gdf_region.plot(
            ax=ax, 
            column="MAR(g/cm**2/ka)", 
            legend=(i==4),  # Only show legend on last plot
            cmap='viridis', 
            markersize=40
        )
    ax.set_title(f"{region_name}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    if i != 4:
        ax.get_legend().remove() if ax.get_legend() else None
fig.suptitle("MAR(g/cm**2/ka) by Region", fontsize=18)
plt.show()

# Optionally, show descriptive statistics for each region
for region_name, bounds in regions.items():
    df_region = df_1[region_mask(df_1, bounds)]
    print(f"Region: {region_name}, Number of points: {len(df_region)}")
    print(df_region.describe())
    print()

### df_2

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file2 = "sedRateData_timo_skagarrak.xlsx"

In [ ]:
df_2 = pd.read_excel(Path(data_path / sedrate_labels_file2))

In [ ]:
df_2

In [ ]:
df_2 = df_2[["Lat N","Long E","Sedimentation rate (cm/yr)"]]
df_2 = df_2[df_2["Long E"]>-180]
df_2["Sedimentation rate (cm/yr)"] = df_2["Sedimentation rate (cm/yr)"]*1e3
df_2 = df_2.dropna(how="any")

In [ ]:
df_2 = df_2.rename(columns={
    "Lat N": "Latitude",
    "Long E": "Longitude",
    "Sedimentation rate (cm/yr)": "Sedimentation rate [cm/ka]"
})

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_2 = df_2.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_2)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_2 = df_2.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_2)

In [ ]:
df_2

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_2.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_2["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_2.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_2['Longitude'], df_2['Latitude'])]
gdf_2 = GeoDataFrame(df_2, geometry=geometry)  
gdf_2.plot(ax = world.plot(figsize = (20, 12)), column = df_2["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_2['Longitude'], df_2['Latitude'])]
gdf_2 = GeoDataFrame(df_2, geometry=geometry)  
gdf_2.plot(ax = world.plot(figsize = (20, 12)), column = df_2["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_2.describe()

### df_3

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file3 = "sedRateData_Catubig_1998.csv"

In [ ]:
df_3 = pd.read_csv(Path(data_path / sedrate_labels_file3), sep= "\t")

In [ ]:
df_3

In [ ]:
df_3 = df_3[["Latitude","Longitude","MAR [g/cm**2/ka]"]]

In [ ]:
"""
df_3 = df_3[["Lat N","Long E","Sedimentation rate (cm/yr)"]]
df_3 = df_3[df_3["Long E"]>-180]
df_3["Sedimentation rate (cm/yr)"] = df_3["Sedimentation rate (cm/yr)"]*1e3
df_3 = df_3.dropna(how="any")
"""

In [ ]:

df_3 = df_3.rename(columns={
    "MAR [g/cm**2/ka]": "MAR(g/cm**2/ka)"
})
df_3["MAR(g/cm**2/ka)"] = df_3["MAR(g/cm**2/ka)"]


In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_3 = df_3.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_3)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_3 = df_3.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_3)

In [ ]:
df_3

In [ ]:


latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
sedrate_list = []

for idx, row in df_3.iterrows():
    lat, lon, MAR = row["Latitude"], row["Longitude"], row["MAR(g/cm**2/ka)"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    sedrate = convertMAR2SR(MAR, porosity, ds)
    sedrate_list.append(sedrate)

df_3["Sedimentation rate [cm/ka]"] = sedrate_list

In [ ]:
df_3.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_3['Longitude'], df_3['Latitude'])]
gdf_3 = GeoDataFrame(df_3, geometry=geometry)  
gdf_3.plot(ax = world.plot(figsize = (20, 12)), column = df_3["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_3['Longitude'], df_3['Latitude'])]
gdf_3 = GeoDataFrame(df_3, geometry=geometry)  
gdf_3.plot(ax = world.plot(figsize = (20, 12)), column = df_3["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_3.describe()

### df_4

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file4 = "sedRateData_Hohmann_2024.csv"

In [ ]:
df_4 = pd.read_csv(Path(data_path / sedrate_labels_file4), sep= ",")

In [ ]:
df_4

In [ ]:
df_4 = df_4[["lat","long","S(cm/a)"]]

In [ ]:
"""
df_3 = df_3[["Lat N","Long E","Sedimentation rate (cm/yr)"]]
df_3 = df_3[df_3["Long E"]>-180]
df_3["Sedimentation rate (cm/yr)"] = df_3["Sedimentation rate (cm/yr)"]*1e3
df_3 = df_3.dropna(how="any")
"""

In [ ]:

df_4 = df_4.rename(columns={
    "lat": "Latitude",
    "long": "Longitude",
    "S(cm/a)": "Sedimentation rate [cm/ka]"
})
df_4["Sedimentation rate [cm/ka]"] = df_4["Sedimentation rate [cm/ka]"]*1e3


In [ ]:
df_4

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_4 = df_4.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_4)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_4 = df_4.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_4)

In [ ]:
df_4

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_4.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_4["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_4.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_4['Longitude'], df_4['Latitude'])]
gdf_4 = GeoDataFrame(df_4, geometry=geometry)  
gdf_4.plot(ax = world.plot(figsize = (20, 12)), column = df_4["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_4['Longitude'], df_4['Latitude'])]
gdf_4 = GeoDataFrame(df_4, geometry=geometry)  
gdf_4.plot(ax = world.plot(figsize = (20, 12)), column = df_4["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_4.describe()

### df_5

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file5 = "sedRateData_MAR_Costa_2021.xlsx"

In [ ]:
df_5 = pd.read_excel(Path(data_path / sedrate_labels_file5))

In [ ]:
df_5

In [ ]:
df_5 = df_5[["Lat ˚N","Long ˚E","Mean Holocene (0-5ka) Mass Flux (g/cm2kyr)"]]

In [ ]:
"""
df_3 = df_3[["Lat N","Long E","Sedimentation rate (cm/yr)"]]
df_3 = df_3[df_3["Long E"]>-180]
df_3["Sedimentation rate (cm/yr)"] = df_3["Sedimentation rate (cm/yr)"]*1e3
df_3 = df_3.dropna(how="any")
"""

In [ ]:

df_5 = df_5.rename(columns={
    "Lat ˚N": "Latitude",
    "Long ˚E": "Longitude",
    "Mean Holocene (0-5ka) Mass Flux (g/cm2kyr)": "MAR(g/cm**2/ka)"
})
df_5["MAR(g/cm**2/ka)"] = df_5["MAR(g/cm**2/ka)"]


In [ ]:
df_5

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_5 = df_5.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_5)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_5 = df_5.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_5)

In [ ]:
df_5["Longitude"].max()

In [ ]:
# from -180 to +180, every 0.1°
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
sedrate_list = []

for idx, row in df_5.iterrows():
    lat, lon, MAR = row["Latitude"], row["Longitude"], row["MAR(g/cm**2/ka)"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    #print(lat_idx)
    #print(lon_idx)
    porosity = int(dataset_numpy_porosity_graw[lat_idx-1, lon_idx-1])
    sedrate = convertMAR2SR(MAR, porosity, ds)
    sedrate_list.append(sedrate)

df_5["Sedimentation rate [cm/ka]"] = sedrate_list

In [ ]:
df_5

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_5['Longitude'], df_5['Latitude'])]
gdf_5 = GeoDataFrame(df_5, geometry=geometry)  
gdf_5.plot(ax = world.plot(figsize = (20, 12)), column = df_5["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_5['Longitude'], df_5['Latitude'])]
gdf_5 = GeoDataFrame(df_5, geometry=geometry)  
gdf_5.plot(ax = world.plot(figsize = (20, 12)), column = df_5["Sedimentation rate [cm/ka]"],legend = True);#"Sedimentation rate [cm/ka]"

df_5.describe()

### df_6

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file6 = "sedRateData_Stahl_2004.csv"

In [ ]:
df_6 = pd.read_csv(Path(data_path / sedrate_labels_file6), sep= ",")

In [ ]:
df_6

In [ ]:
df_6 = df_6[["Latitude","Longitude","MAR"]]

In [ ]:
"""
df_3 = df_3[["Lat N","Long E","Sedimentation rate (cm/yr)"]]
df_3 = df_3[df_3["Long E"]>-180]
df_3["Sedimentation rate (cm/yr)"] = df_3["Sedimentation rate (cm/yr)"]*1e3
df_3 = df_3.dropna(how="any")
"""

In [ ]:

df_6 = df_6.rename(columns={
    "MAR": "MAR(g/cm**2/ka)"
})
df_6["MAR(g/cm**2/ka)"] = df_6["MAR(g/cm**2/ka)"]*1e3


In [ ]:
df_6

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_6 = df_6.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_6)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_6 = df_6.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_6)

In [ ]:
df_6

In [ ]:
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
sedrate_list = []

for idx, row in df_6.iterrows():
    lat, lon, MAR = row["Latitude"], row["Longitude"], row["MAR(g/cm**2/ka)"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    sedrate = convertMAR2SR(MAR, porosity, ds)
    sedrate_list.append(sedrate)

df_6["Sedimentation rate [cm/ka]"] = sedrate_list

In [ ]:
df_6.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_6['Longitude'], df_6['Latitude'])]
gdf_6 = GeoDataFrame(df_6, geometry=geometry)  
gdf_6.plot(ax = world.plot(figsize = (20, 12)), column = df_6["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_6['Longitude'], df_6['Latitude'])]
gdf_6 = GeoDataFrame(df_6, geometry=geometry)  
gdf_6.plot(ax = world.plot(figsize = (20, 12)), column = df_6["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_6.describe()

### df_7

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file7 = "sedRateData_diesing_2021.xlsx"

In [ ]:
df_7 = pd.read_excel(Path(data_path / sedrate_labels_file7))

In [ ]:
df_7

In [ ]:
df_7 = df_7[["Latitude","Longitude","SedRate"]]
df_7 = df_7[df_7["Longitude"]>-180]
df_7["SedRate"] = df_7["SedRate"]*1e3
df_7 = df_7.dropna(how="any")

In [ ]:
df_7 = df_7.rename(columns={
    "SedRate": "Sedimentation rate [cm/ka]"
})

In [ ]:
# 1. Extract rows with Sedimentation rate = 0.01
df_zero = df_7[df_7["Sedimentation rate [cm/ka]"] == 0.01]

# 2. Keep every 10th row (0, 10, 20, 30, ...)
df_zero_reduced = df_zero.iloc[::5]

# 3. Keep all other points normally
df_other = df_7[df_7["Sedimentation rate [cm/ka]"] != 0.01]

# 4. Combine reduced cluster + remaining points
df_thinned = pd.concat([df_other, df_zero_reduced], ignore_index=True)

# 5. Optional: sort by index or coordinates if needed
df_thinned = df_thinned.sort_values(["Latitude", "Longitude"]).reset_index(drop=True)

north_sea_bounds = {
    "lon_min": -5,
    "lon_max": 10,
    "lat_min": 51,
    "lat_max": 62
}





In [ ]:
df_7 = df_thinned
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_7 = df_7.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_7)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_7 = df_7.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_7)

In [ ]:
df_7

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_7.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_7["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_7.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_7['Longitude'], df_7['Latitude'])]
gdf_7 = GeoDataFrame(df_7, geometry=geometry)  
gdf_7.plot(ax = world.plot(figsize = (20, 12)), column = df_7["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_7['Longitude'], df_7['Latitude'])]
gdf_7 = GeoDataFrame(df_7, geometry=geometry)  
gdf_7.plot(ax = world.plot(figsize = (20, 12)), column = df_7["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_7.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10), dpi=200)

# Plot world boundaries
world.plot(ax=ax, color="lightgray", edgecolor="white")

# Plot only points in the North Sea region
gdf_7.plot(
    ax=ax,
    column=df_7["Sedimentation rate [cm/ka]"],  # or MAR(g/cm**2/ka)
    cmap="viridis",
    vmin=0,
    vmax=50,
    legend=True,
    markersize=20,
    alpha=0.8
)

# Set map limits (zooms into North Sea)
ax.set_xlim(north_sea_bounds["lon_min"], north_sea_bounds["lon_max"])
ax.set_ylim(north_sea_bounds["lat_min"], north_sea_bounds["lat_max"])

ax.set_title("North Sea – Sedimentation Rates", fontsize=16)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()


In [ ]:
df_7

### df_8

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file8 = "sedRateData_pandarinath_et_al_2004.xlsx"

In [ ]:
df_8 = pd.read_excel(Path(data_path / sedrate_labels_file8))

In [ ]:
df_8

In [ ]:
df_8 = df_8[["Latitude","Longitude","Sedimentation rate (mm/yr).1"]]
df_8 = df_8[df_8["Longitude"]>-180]
df_8["Sedimentation rate (mm/yr).1"] = df_8["Sedimentation rate (mm/yr).1"]*100
df_8 = df_8.dropna(how="any")

In [ ]:
df_8 = df_8.rename(columns={
    "Sedimentation rate (mm/yr).1": "Sedimentation rate [cm/ka]"
})
#df_8["Sedimentation rate [cm/ka]"] = df_8["Sedimentation rate [cm/ka]"]*100


In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_8 = df_8.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")

print(df_8)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_8 = df_8.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_8)

In [ ]:
df_8

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_8.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_8["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_8

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_8['Longitude'], df_8['Latitude'])]
gdf_8 = GeoDataFrame(df_8, geometry=geometry)  
gdf_8.plot(ax = world.plot(figsize = (20, 12)), column = df_8["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_8['Longitude'], df_8['Latitude'])]
gdf_8 = GeoDataFrame(df_8, geometry=geometry)  
gdf_8.plot(ax = world.plot(figsize = (20, 12)), column = df_8["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_8.describe()

### df_9

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file9 = "sedRateData_srisuksawad_et_al_1997.xlsx"

In [ ]:
df_9 = pd.read_excel(Path(data_path / sedrate_labels_file9))

In [ ]:
df_9

In [ ]:
df_9 = df_9[["Decimal Latitude","Decimal Longitude","SR @ Interface (cm/yr)"]]
df_9 = df_9[df_9["Decimal Longitude"]>-180]
df_9["SR @ Interface (cm/yr)"] = df_9["SR @ Interface (cm/yr)"]*1000
df_9 = df_9.dropna(how="any")

In [ ]:
df_9 = df_9.rename(columns={
    "SR @ Interface (cm/yr)": "Sedimentation rate [cm/ka]",
    "Decimal Latitude": "Latitude",
    "Decimal Longitude": "Longitude"
})
#df_8["Sedimentation rate [cm/ka]"] = df_8["Sedimentation rate [cm/ka]"]*100


In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_9 = df_9.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_9)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_9 = df_9.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_9)

In [ ]:
df_9

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_9.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_9["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_9.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_9['Longitude'], df_9['Latitude'])]
gdf_9 = GeoDataFrame(df_9, geometry=geometry)  
gdf_9.plot(ax = world.plot(figsize = (20, 12)), column = df_9["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_9['Longitude'], df_9['Latitude'])]
gdf_9 = GeoDataFrame(df_9, geometry=geometry)  
gdf_9.plot(ax = world.plot(figsize = (20, 12)), column = df_9["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_9.describe()

### df_10

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file10 = "sedRateData_Munoz_et_al_2004.xlsx"

In [ ]:
df_10 = pd.read_excel(Path(data_path / sedrate_labels_file10), sheet_name="Sheet2")

In [ ]:
df_10

In [ ]:
MAR_list = df_10["Accum. Rate (g/cm²/yr) "]*1000
df_10 = df_10[["Decimal Latitude","Decimal Longitude","SR (cm/yr)"]]
df_10 = df_10[df_10["Decimal Longitude"]>-180]
df_10["SR (cm/yr)"] = df_10["SR (cm/yr)"]*1000
df_10 = df_10.dropna(how="any")


In [ ]:
df_10 = df_10.rename(columns={
    "SR (cm/yr)": "Sedimentation rate [cm/ka]",
    "Decimal Latitude": "Latitude",
    "Decimal Longitude": "Longitude"
})
#df_8["Sedimentation rate [cm/ka]"] = df_8["Sedimentation rate [cm/ka]"]*100


In [ ]:
df_10 = df_10.drop(index=[1])
MAR_list = MAR_list.drop(index=[1])

In [ ]:
df_10

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_10 = df_10.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_10)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_10 = df_10.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_10)

In [ ]:
df_10["Sedimentation rate [cm/ka]"] = pd.to_numeric(df_10["Sedimentation rate [cm/ka]"], errors='coerce')

In [ ]:
"""
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_10.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)
"""

# Add MAR to the DataFrame
df_10["MAR(g/cm**2/ka)"] = MAR_list


In [ ]:
df_10.head(n=10)

In [ ]:
df_10["MAR(g/cm**2/ka)"] = pd.to_numeric(df_10["MAR(g/cm**2/ka)"], errors='coerce')

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_10['Longitude'], df_10['Latitude'])]
gdf_10 = GeoDataFrame(df_10, geometry=geometry)  
gdf_10.plot(ax = world.plot(figsize = (20, 12)), column = df_10["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_10['Longitude'], df_10['Latitude'])]
gdf_10 = GeoDataFrame(df_10, geometry=geometry)  
gdf_10.plot(ax = world.plot(figsize = (20, 12)), column = df_10["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_10.describe()

### df_11

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file11 = "sedRateData_singh_et_al_2017.xlsx"

In [ ]:
df_11 = pd.read_excel(Path(data_path / sedrate_labels_file11), sheet_name="Sheet2")

In [ ]:
df_11

In [ ]:
df_11 = df_11[["Latitude (N)","Longitude (E)","Sedimentation rate (cm/ka)"]]
df_11 = df_11[df_11["Longitude (E)"]>-180]
#df_11["Sedimentation rate (cm/ka)"] = df_11["Sedimentation rate (cm/ka)"]*1000
df_11 = df_11.dropna(how="any")

In [ ]:
df_11 = df_11.rename(columns={
    "Sedimentation rate (cm/ka)": "Sedimentation rate [cm/ka]",
    "Latitude (N)": "Latitude",
    "Longitude (E)": "Longitude"
})
#df_8["Sedimentation rate [cm/ka]"] = df_8["Sedimentation rate [cm/ka]"]*100


In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_11 = df_11.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_11)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_11 = df_11.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_11)

In [ ]:
df_11

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_11.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_11["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_11.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_11['Longitude'], df_11['Latitude'])]
gdf_11 = GeoDataFrame(df_11, geometry=geometry)  
gdf_11.plot(ax = world.plot(figsize = (20, 12)), column = df_11["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_11['Longitude'], df_11['Latitude'])]
gdf_11 = GeoDataFrame(df_11, geometry=geometry)  
gdf_11.plot(ax = world.plot(figsize = (20, 12)), column = df_11["Sedimentation rate [cm/ka]"],legend = True);#"Sementation rate [cm/ka]"

df_11.describe()

### df_12

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file12 = "sedRateData_oguri_et_al_2022.xlsx"

In [ ]:
df_12 = pd.read_excel(Path(data_path / sedrate_labels_file12), sheet_name="Sheet1")

In [ ]:
df_12

In [ ]:
MAR_list = df_12["MAR (g\xa0cm−2\xa0y−1)"]
df_12 = df_12[["Decimal Latitude","Decimal Longitude","SR (mm\xa0y−1)"]]
df_12 = df_12[df_12["Decimal Longitude"]>-180]
df_12["SR (mm\xa0y−1)"] = df_12["SR (mm\xa0y−1)"]*100
df_12 = df_12.dropna(how="any")

In [ ]:
df_12.columns

In [ ]:
df_12 = df_12.rename(columns={
    "SR (mm\xa0y−1)": "Sedimentation rate [cm/ka]",
    "Decimal Latitude": "Latitude",
    "Decimal Longitude": "Longitude"
})
#df_8["Sedimentation rate [cm/ka]"] = df_8["Sedimentation rate [cm/ka]"]*100


In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_12 = df_12.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_12)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_12 = df_12.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_12)

In [ ]:
df_12

In [ ]:
"""
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_12.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)
"""
# Add MAR to the DataFrame
df_12["MAR(g/cm**2/ka)"] = MAR_list
df_12["MAR(g/cm**2/ka)"] = df_12["MAR(g/cm**2/ka)"]*1e3

In [ ]:
df_12.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_12['Longitude'], df_12['Latitude'])]
gdf_12 = GeoDataFrame(df_12, geometry=geometry)  
gdf_12.plot(ax = world.plot(figsize = (20, 12)), column = df_12["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_12['Longitude'], df_12['Latitude'])]
gdf_12 = GeoDataFrame(df_12, geometry=geometry)  
gdf_12.plot(ax = world.plot(figsize = (20, 12)), column = df_12["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_12.describe()

### df_13

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file13 = "sedRateData_Ferreira_et_al_2020.xlsx"

In [ ]:
df_13 = pd.read_excel(Path(data_path / sedrate_labels_file13), sheet_name="Sheet3")

In [ ]:
df_13

In [ ]:
df_13.columns

In [ ]:
df_13 = df_13[["Latitude (°S)","Longitude (°W)","Sedimentation Rate (cm ka⁻¹)"]]
df_13 = df_13[df_13["Longitude (°W)"]>-180]
df_13["Sedimentation Rate (cm ka⁻¹)"] = df_13["Sedimentation Rate (cm ka⁻¹)"]
df_13["Latitude (°S)"] = df_13["Latitude (°S)"]*-1
df_13["Longitude (°W)"] = df_13["Longitude (°W)"]*-1
df_13 = df_13.dropna(how="any")

In [ ]:
df_13 = df_13.rename(columns={
    "Sedimentation Rate (cm ka⁻¹)": "Sedimentation rate [cm/ka]",
    "Latitude (°S)": "Latitude",
    "Longitude (°W)": "Longitude"
})
#df_8["Sedimentation rate [cm/ka]"] = df_8["Sedimentation rate [cm/ka]"]*100


In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_13 = df_13.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_13)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_13 = df_13.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_13)

In [ ]:
df_13

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_13.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_13["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_13.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_13['Longitude'], df_13['Latitude'])]
gdf_13 = GeoDataFrame(df_13, geometry=geometry)  
gdf_13.plot(ax = world.plot(figsize = (20, 12)), column = df_13["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_13['Longitude'], df_13['Latitude'])]
gdf_13 = GeoDataFrame(df_13, geometry=geometry)  
gdf_13.plot(ax = world.plot(figsize = (20, 12)), column = df_13["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_13.describe()

### df_14

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file14 = "sedRateData_Mollenhauer_et_al_2002.xlsx"

In [ ]:
df_14 = pd.read_excel(Path(data_path / sedrate_labels_file14), sheet_name="Sheet3")

In [ ]:
df_14

In [ ]:
df_14 = df_14[["Latitude (°N)","Longitude (°E)","Sedimentation Rate (cm kyr⁻¹)"]]
df_14 = df_14[df_14["Longitude (°E)"]>-180]
#df_14["Sedimentation rate (cm/ka)"] = df_14["Sedimentation rate (cm/ka)"]*1000
df_14 = df_14.dropna(how="any")

In [ ]:
df_14 = df_14.rename(columns={
    "Sedimentation Rate (cm kyr⁻¹)": "Sedimentation rate [cm/ka]",
    "Latitude (°N)": "Latitude",
    "Longitude (°E)": "Longitude"
})
#df_8["Sedimentation rate [cm/ka]"] = df_8["Sedimentation rate [cm/ka]"]*100


In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_14 = df_14.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_14)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_14 = df_14.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_14)

In [ ]:
df_14

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_14.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_14["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_14.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_14['Longitude'], df_14['Latitude'])]
gdf_14 = GeoDataFrame(df_14, geometry=geometry)  
gdf_14.plot(ax = world.plot(figsize = (20, 12)), column = df_14["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_14['Longitude'], df_14['Latitude'])]
gdf_14 = GeoDataFrame(df_14, geometry=geometry)  
gdf_14.plot(ax = world.plot(figsize = (20, 12)), column = df_14["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_14.describe()

### df_15

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file15 = "combined_mar_points_wegner_2015.csv"

In [ ]:
df_15 = pd.read_csv(Path(data_path / sedrate_labels_file15), sep= ",")

In [ ]:
df_15

In [ ]:
#df_6 = df_6[["lat","lon","SR"]]

In [ ]:
#"""
df_15 = df_15[["lat","lon","SR(cm/ka)"]]
df_15 = df_15[df_15["lon"]>-180]
df_15 = df_15[df_15["lon"]<180]
df_15 = df_15[df_15["lat"]>-90]
df_15 = df_15[df_15["lat"]<90]
#df_15["Sedimentation rate (cm/yr)"] = df_15["Sedimentation rate (cm/yr)"]*1e3
df_15 = df_15.dropna(how="any")
#"""

In [ ]:

df_15 = df_15.rename(columns={
    "SR(cm/ka)": "Sedimentation rate [cm/ka]", 
    "lat": "Latitude", 
    "lon" : "Longitude"
})
#df_6["Sedimentation rate [cm/ka]"] = df_6["Sedimentation rate [cm/ka]"]*1e3


In [ ]:
df_15

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_15 = df_15.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_15)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_15 = df_15.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_15)

In [ ]:
#df_15["Longitude"].max()

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_15.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_15["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_15.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_15['Longitude'], df_15['Latitude'])]
gdf_15 = GeoDataFrame(df_15, geometry=geometry)  
gdf_15.plot(ax = world.plot(figsize = (20, 12)), column = df_15["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_15['Longitude'], df_15['Latitude'])]
gdf_15 = GeoDataFrame(df_15, geometry=geometry)  
gdf_15.plot(ax = world.plot(figsize = (20, 12)), column = df_15["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_15.describe()

### df_16

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file16 = "combined_mar_points_deng_2006_filtered.csv"

In [ ]:
df_16 = pd.read_csv(Path(data_path / sedrate_labels_file16), sep= ",")

In [ ]:
df_16

In [ ]:
#df_6 = df_6[["lat","lon","SR"]]

In [ ]:
#"""
df_16 = df_16[["lat","lon","MAR(g/cm2/yr)"]]
df_16 = df_16[df_16["lon"]>-180]
df_16 = df_16[df_16["lon"]<180]
df_16["MAR(g/cm2/yr)"] = df_16["MAR(g/cm2/yr)"]*1000
#df_16["Sedimentation rate (cm/yr)"] = df_16["Sedimentation rate (cm/yr)"]*1e3
df_16 = df_16.dropna(how="any")
#"""

In [ ]:

df_16 = df_16.rename(columns={
    "MAR(g/cm2/yr)": "MAR(g/cm**2/ka)", 
    "lat": "Latitude", 
    "lon" : "Longitude"
})
#df_6["Sedimentation rate [cm/ka]"] = df_6["Sedimentation rate [cm/ka]"]*1e3


In [ ]:
df_16

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_16 = df_16.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_16)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_16 = df_16.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_16)

In [ ]:
#df_15["Longitude"].max()

In [ ]:
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
sedrate_list = []

for idx, row in df_16.iterrows():
    lat, lon, MAR = row["Latitude"], row["Longitude"], row["MAR(g/cm**2/ka)"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    sedrate = convertMAR2SR(MAR, porosity, ds)
    sedrate_list.append(sedrate)

df_16["Sedimentation rate [cm/ka]"] = sedrate_list

In [ ]:
df_16.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_16['Longitude'], df_16['Latitude'])]
gdf_16 = GeoDataFrame(df_16, geometry=geometry)  
gdf_16.plot(ax = world.plot(figsize = (20, 12)), column = df_16["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_16['Longitude'], df_16['Latitude'])]
gdf_16 = GeoDataFrame(df_16, geometry=geometry)  
gdf_16.plot(ax = world.plot(figsize = (20, 12)), column = df_16["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_16.describe()

### df_17

In [ ]:
#data_path = Path().resolve().parents[3] / "work_old/work/toc-neural-nets/SelectFeaturesRun/pyfiles/" 
sedrate_labels_file17 = "south_china_sea_sheng_et_al_2024.csv"

In [ ]:
df_17 = pd.read_csv(Path(data_path / sedrate_labels_file17), sep= ",")

In [ ]:
df_17["SR(cm/ka)"][6508]+2

In [ ]:
#df_6 = df_6[["lat","lon","SR"]]

In [ ]:
#"""
df_17 = df_17[["lat","lon","SR(cm/ka)"]]
df_17 = df_17[df_17["lon"]>-180]
df_17 = df_17[df_17["lon"]<180]
df_17 = df_17[df_17["lat"]>-90]
df_17 = df_17[df_17["lat"]<90]
#df_17["SR(cm/ka)"] = float(df_17["SR(cm/ka)"])#*1e3
df_17 = df_17.dropna(how="any")
#"""

In [ ]:

df_17 = df_17.rename(columns={
    "SR(cm/ka)": "Sedimentation rate [cm/ka]", 
    "lat": "Latitude", 
    "lon" : "Longitude"
})
#df_6["Sedimentation rate [cm/ka]"] = df_6["Sedimentation rate [cm/ka]"]*1e3


In [ ]:
df_17

In [ ]:
import pandas as pd

# Group by Latitude and Longitude and filter groups with more than one entry
df_17 = df_17.groupby(['Latitude', 'Longitude']).filter(lambda x: len(x) > 0)

# Display the filtered DataFrame
print("Filtered DataFrame:")
print(df_17)

# Group the data by Latitude and Longitude and calculate the mean of each group
df_17 = df_17.groupby(['Latitude', 'Longitude'], as_index=False).mean()

# Display the resulting DataFrame with averaged locations
print("DataFrame after averaging:")
print(df_17)

In [ ]:
#df_15["Longitude"].max()

In [ ]:
MAR_list = []
latitudes = np.linspace(-90, 90, 2161)      # from -90 to +90, every 0.1°
longitudes = np.linspace(-180, 180, 4321)   # from -180 to +180, every 0.1°
for idx, row in df_17.iterrows():
    lat, lon, sedrate = row["Latitude"], row["Longitude"], row["Sedimentation rate [cm/ka]"]
    lat_idx = find_nearest(lat, latitudes)
    lon_idx = find_nearest(lon, longitudes)
    porosity = int(dataset_numpy_porosity_graw[lat_idx, lon_idx])
    MAR = convertSR2MAR(sedrate, porosity, ds)
    MAR_list.append(MAR)

# Add MAR to the DataFrame
df_17["MAR(g/cm**2/ka)"] = MAR_list

In [ ]:
df_17.head(n=10)

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_17['Longitude'], df_17['Latitude'])]
gdf_17 = GeoDataFrame(df_17, geometry=geometry)  
gdf_17.plot(ax = world.plot(figsize = (20, 12)), column = df_17["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_17['Longitude'], df_17['Latitude'])]
gdf_17 = GeoDataFrame(df_17, geometry=geometry)  
gdf_17.plot(ax = world.plot(figsize = (20, 12)), column = df_17["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

df_17.describe()

In [ ]:
# Plot only data for Chinese seas in log scale

# Define Chinese seas bounding box (approximate extent)
# South China Sea, East China Sea, Yellow Sea, Bohai Sea
lon_min, lon_max = 90, 130   # degrees East
lat_min, lat_max = 0, 42     # degrees North

# Filter the DataFrame for Chinese seas
df_17_cn = df_17[
    (df_17["Longitude"] >= lon_min) & (df_17["Longitude"] <= lon_max) &
    (df_17["Latitude"] >= lat_min) & (df_17["Latitude"] <= lat_max)
].copy()

# Prepare geometry and GeoDataFrame for plotting
geometry = [Point(xy) for xy in zip(df_17_cn['Longitude'], df_17_cn['Latitude'])]
gdf_17_cn = GeoDataFrame(df_17_cn, geometry=geometry)

# Read the world map and clip to the Chinese seas bounding box for clarity
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
bbox = world.cx[lon_min:lon_max, lat_min:lat_max]

# Plot MAR(g/cm**2/ka) in log scale
ax = bbox.plot(figsize=(20, 12), color="white", edgecolor="black")
gdf_17_cn.plot(
    ax=ax, 
    column="MAR(g/cm**2/ka)", 
    legend=True, 
    cmap='viridis', 
    norm=plt.matplotlib.colors.LogNorm(vmin=gdf_17_cn["MAR(g/cm**2/ka)"].replace(0, np.nan).min()+1e-6, vmax=gdf_17_cn["MAR(g/cm**2/ka)"].max()) # avoid 0 for log
)
plt.title("MAR(g/cm²/ka) in Chinese seas (log scale)")
plt.xlim(lon_min, lon_max)
plt.ylim(lat_min, lat_max)
plt.show()

# Plot Sedimentation rate [cm/ka] in log scale
ax = bbox.plot(figsize=(20, 12), color="white", edgecolor="black")
gdf_17_cn.plot(
    ax=ax, 
    column="Sedimentation rate [cm/ka]", 
    legend=True, 
    cmap='plasma', 
    norm=plt.matplotlib.colors.LogNorm(vmin=gdf_17_cn["Sedimentation rate [cm/ka]"].replace(0, np.nan).min()+1e-6, vmax=gdf_17_cn["Sedimentation rate [cm/ka]"].max()) # avoid 0 for log
)
plt.title("Sedimentation rate [cm/ka] in Chinese seas (log scale)")
plt.xlim(lon_min, lon_max)
plt.ylim(lat_min, lat_max)
plt.show()

df_17_cn.describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Load the array (2160, 4320)
mask = np.load(str(Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/features/FeaturesPhrampusLee_TOCSedRate_updated/stone_gravel_mask_5arcmin.npy"))

plt.figure(figsize=(16, 8))
plt.imshow(mask, origin="upper", cmap="viridis", interpolation="none", aspect="auto")
plt.title("Stone/Gravel Mask (5 arcmin resolution)")
plt.xlabel("Longitude pixel")
plt.ylabel("Latitude pixel")
plt.colorbar(label="Mask value")
plt.show()

# Get number of grid points where value==1
num_mask_ones = np.sum(mask == 1)
print(f"Number of grid points where mask == 1: {num_mask_ones}")

# Compute lat and lon for each grid cell (assuming full global grid from -90 to 90, -180 to 180)
n_lat, n_lon = mask.shape
lat_step = 180 / n_lat
lon_step = 360 / n_lon

# Center of each pixel in 5 arcmin resolution
lat_vals = np.linspace(90 - 0.5*lat_step, -90 + 0.5*lat_step, n_lat)
lon_vals = np.linspace(-180 + 0.5*lon_step, 180 - 0.5*lon_step, n_lon)

# Get where mask==1
mask_indices = np.argwhere(mask == 1)

# Downsample: Select every 100th grid point to get a low resolution sample
downsample_indices = mask_indices[::100]
latitudes = lat_vals[downsample_indices[:, 0]]
longitudes = lon_vals[downsample_indices[:, 1]]

# Hudson Bay bounding box (approximate): Lat 50–65N, Lon -95–-75E (converted for this grid convention)
hb_lat_min, hb_lat_max = 50.0, 75.0
hb_lon_min, hb_lon_max = -105.0, -65.0

# Remove points in Hudson Bay
hb_mask = ~(
    (latitudes >= hb_lat_min) & (latitudes <= hb_lat_max) &
    (longitudes >= hb_lon_min) & (longitudes <= hb_lon_max)
)
latitudes_filtered = latitudes[hb_mask]
longitudes_filtered = longitudes[hb_mask]

# Set MAR and Sedimentation rate to zero for all selected points outside Hudson Bay
mar_zero = np.full_like(latitudes_filtered, 1e-5, dtype=float)
sedrate_zero = np.full_like(latitudes_filtered, 1e-5, dtype=float)

# Create DataFrame
df_gravel_mask = pd.DataFrame({
    "Latitude": latitudes_filtered,
    "Longitude": longitudes_filtered,
    "MAR(g/cm**2/ka)": mar_zero,
    "Sedimentation rate [cm/ka]": sedrate_zero,
})
print(df_gravel_mask.head())
df_18 = df_gravel_mask

In [ ]:
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_18['Longitude'], df_18['Latitude'])]
gdf_18 = GeoDataFrame(df_18, geometry=geometry)  
gdf_18.plot(ax = world.plot(figsize = (20, 12)), column = df_18["MAR(g/cm**2/ka)"],legend = True);#"Sedimentation rate [cm/ka]"
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
geometry = [Point(xy) for xy in zip(df_18['Longitude'], df_18['Latitude'])]
gdf_18 = GeoDataFrame(df_18, geometry=geometry)  
gdf_18.plot(ax = world.plot(figsize = (20, 12)), column = df_18["Sedimentation rate [cm/ka]"], vmin = 0, vmax = 5000,legend = True);#"Sedimentation rate [cm/ka]"

In [ ]:
df_18

In [ ]:
df_list = [df_1, df_2, df_3, df_4, df_6, df_7, df_8, df_9, df_10, df_11, df_12, df_13, df_14, df_15, df_16, df_17, df_18]
for i in range(17):
    df_list[i]["Sedimentation rate [cm/ka]"].max()
    df_list[i]["Sedimentation rate [cm/ka]"] = pd.to_numeric(df_list[i]["Sedimentation rate [cm/ka]"])
    

In [ ]:
df = pd.concat([df_1, df_2, df_3, df_4, df_5, df_6, df_7, df_8, df_9, df_10, df_11, df_12, df_13, df_14, df_15, df_16, df_17, df_18])

In [ ]:
df

In [ ]:
#df_with_15 = pd.concat([df_5, df])

In [ ]:
# Create figure
plt.figure(figsize=(5, 5))

# Plot histogram and KDE for df
df["MAR(g/cm**2/ka)"].hist(bins=500, density=True, alpha=0.3, color='blue', label='df Histogram')
df["MAR(g/cm**2/ka)"].plot.kde(color='blue', label='df KDE')

# Plot histogram and KDE for df_15
df_15["MAR(g/cm**2/ka)"].hist(bins=500, density=True, alpha=0.3, color='red', label='Th based Histogram')
df_15["MAR(g/cm**2/ka)"].plot.kde(color='red', label='Th based KDE')

plt.xlabel('MAR (g/cm²/ka)')
plt.ylabel('Density')
plt.title('Distribution of MAR, comparing Th-based and other isotopes')
plt.xlim(-5, 500)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cmcrameri.cm as cmc
import cmocean as cmo

# Load world boundaries
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Remove duplicates
df = df.drop_duplicates()

# Create GeoDataFrame for data points
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf = GeoDataFrame(df, geometry=geometry, crs=world.crs)

# Remove points that lie on land
# Use spatial join to find which points are within any land polygon
gdf_land = gpd.sjoin(gdf, world, how="left", predicate="within")
# Points on land will have a non-null index_right (joined to a country polygon)
gdf_sea = gdf_land[gdf_land["index_right"].isnull()].copy()

# Set up figure and axes
fig, ax = plt.subplots(figsize=(20, 10), dpi=300)

# Set a visually appealing background color (e.g., very light blue)
ax.set_facecolor('#e6f2ff')  # pastel blue

# Plot world boundaries with white fill and light gray edges for contrast
world.plot(ax=ax, alpha=1, color='white', edgecolor='white', linewidth=0.5, zorder=1)

# Plot only the points that are not on land
cax = gdf_sea.plot(
    ax=ax,
    column=gdf_sea["MAR(g/cm**2/ka)"],
    cmap=cmo.cm.thermal,
    norm=LogNorm(vmin=1e-1, vmax=5000),
    legend=True,
    markersize=10,
    marker='o',
    #edgecolor='black',
    linewidth=0.2,
    alpha=0.85,
    legend_kwds={
        "shrink": 0.75,
        "label": "MAR (g/cm²/ka)",
        "pad": 0.01
    },
    zorder=2
)

# Set font size for all elements
plt.xlabel("Longitude", fontsize=20)
plt.ylabel("Latitude", fontsize=20)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.xlim(-180, 180)
plt.ylim(-90, 90)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Load world boundaries
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Remove duplicates
df = df.drop_duplicates()

# Create GeoDataFrame for data points
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf = GeoDataFrame(df, geometry=geometry, crs=world.crs)

# Remove points that lie on l
gdf_land = gpd.sjoin(gdf, world, how="left", predicate="within")
gdf_sea = gdf_land[gdf_land["index_right"].isnull()].copy()

# Set up figure and axes with black background
fig, ax = plt.subplots(figsize=(20, 10), dpi=300)
ax.set_facecolor('black')  # black background

# Plot world boundaries with black fill and gray edges for contrast
world.plot(ax=ax, alpha=1, color='gray', edgecolor='gray', linewidth=0.5, zorder=1)

# Plot only the points that are not on land, using cubehelix colormap
cax = gdf_sea.plot(
    ax=ax,
    column=gdf_sea["MAR(g/cm**2/ka)"],
    cmap='cmc.hawaii',
    norm=LogNorm(vmin=1e-1, vmax=5000),
    legend=True,
    markersize=10,
    marker='o',
    linewidth=0.2,
    alpha=0.85,
    legend_kwds={
        "shrink": 0.75,
        "label": "MAR (g/cm²/ka)",
        "pad": 0.01
    },
    zorder=2
)

# Set font size for all elements and white font color for axes labels/ticks
ax.set_xlabel("Longitude", fontsize=20, color='black')
ax.set_ylabel("Latitude", fontsize=20, color='black')
ax.tick_params(axis='x', colors='black', labelsize=20)
ax.tick_params(axis='y', colors='black', labelsize=20)
plt.xlim(-180, 180)
plt.ylim(-90, 90)
plt.tight_layout()

# Set legend text color to white for visibility on black background
leg = ax.get_legend()
if leg is not None:
    for text in leg.get_texts():
        text.set_color('black')
    leg.get_title().set_color('black')
    leg.get_frame().set_facecolor('black')
    leg.get_frame().set_edgecolor('gray')

plt.show()


In [ ]:
"""
# Filter TOC data for Baltic Sea region
baltic_df = df[
    (df['Latitude'] >= 53) & (df['Latitude'] <= 66) &
    (df['Longitude'] >= 12) & (df['Longitude'] <= 30)
].copy()

print(f"Number of points in Baltic Sea dataset: {len(baltic_df)}")
"""

In [ ]:
"""
# Create a GeoDataFrame for the Baltic Sea
geometry_baltic = [Point(xy) for xy in zip(baltic_df['Longitude'], baltic_df['Latitude'])]
gdf_baltic = GeoDataFrame(baltic_df, geometry=geometry_baltic)
"""

In [ ]:
"""
# Plotting Baltic Sea TOC
fig, ax = plt.subplots(figsize=(12, 8), dpi=300)

# Background color
ax.set_facecolor('black')

# Plot TOC points
cax = gdf_baltic.plot(
    ax=ax,
    column=baltic_df["MAR(g/cm**2/ka)"],
    cmap='cmc.hawaii',
    norm=LogNorm(vmin=1e-1, vmax=5000),
    legend=True,
    markersize=10,
    marker='o',
    linewidth=0.2,
    alpha=0.85,
    legend_kwds={
        "shrink": 0.75,
        "label": "MAR (g/cm²/ka)",
        "pad": 0.01
    }
)

# Plot world map to give context
world.plot(ax=ax, alpha=0.5, color='grey')

# Set limits to Baltic Sea region
plt.xlim(10, 30)
plt.ylim(53, 66)

# Labels and formatting
plt.xlabel("Longitude", fontsize=16)
plt.ylabel("Latitude", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title("MAR Data in the Baltic Sea", fontsize=18)
plt.tight_layout()

plt.show()
"""

In [ ]:
"""
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cmcrameri.cm as cmc

# Load the ocean GeoDataFrame
ocean_gdf = gpd.read_file(project_root / "data" / "interim" / "GOaS_v1_20211214" / "goas_v01.shp")

# Filter for the Baltic Sea region
baltic_sea_filtered = ocean_gdf[ocean_gdf["name"] == "Baltic Sea"]
# Load the Baltic Sea shapefile
#baltic_sea = gpd.read_file(str(Path().resolve().parents[2] / "nn-toc-update" / "data/Baltic sea/World_Seas_IHO_v3.shp"))
#baltic_sea = baltic_sea[baltic_sea["NAME"].isin(["Baltic Sea", "Gulf of Finland", "Gulf of Bothnia", "Gulf of Riga"])]
#baltic_sea_filtered = baltic_sea[baltic_sea["NAME"].isin(["Baltic Sea", "Gulf of Finland", "Gulf of Bothnia" , "Gulf of Riga"])]

# Assuming 'df' is your DataFrame with Latitude, Longitude, and TOC columns
df = baltic_df.drop_duplicates()

# Create a GeoDataFrame for the points
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf = GeoDataFrame(df, geometry=geometry)

# Ensure both GeoDataFrames are in the same coordinate reference system (CRS)
gdf = gdf.set_crs("EPSG:4326")  # Coordinate reference system: WGS84
baltic_sea_filtered = baltic_sea_filtered.set_crs("EPSG:4326")

# Perform a spatial join to get only points inside the Baltic Sea region
gdf_baltic = gpd.sjoin(gdf, baltic_sea_filtered, how="inner", op="within")

# Plotting
fig, ax = plt.subplots(figsize=(20, 10), dpi=300)

# Plotting the Baltic Sea region (Water)
baltic_sea_filtered.plot(ax=ax, color='gray', edgecolor='lightgray', alpha=0.7)


# Plotting the points inside the Baltic Sea region
cax = gdf_baltic.plot(ax=ax, column="MAR(g/cm**2/ka)",cmap='cmc.hawaii',
    norm=LogNorm(vmin=1e-1, vmax=5000),
                      legend=True, markersize=5, legend_kwds={
    "shrink": 0.75,
    "label": "MAR(g/cm**2/ka)",
    "pad": 0.01
})

# Customize the plot
ax.set_title("MAR Data Points in the Baltic Sea", fontsize=20)
ax.set_xlabel("Longitude", fontsize=15)
ax.set_ylabel("Latitude", fontsize=15)
ax.set_facecolor('black')  # Set background color to white for contrast

# Adjust layout
plt.tight_layout()
plt.show()
"""

In [ ]:
#df.shape

In [ ]:
#max(df["Sedimentation rate [cm/ka]"])

In [ ]:
#feature_image_path ="/media/nparameswaran/EXTERNAL_USB/LeeKNNsTOC(Data)/RestrepoFeatureMaps/"
restrepo_features_path = Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/features/FeaturesPhrampusLee_TOCSedRate_updated"
#restrepo_features_path_flipped = Path("/media/nparameswaran/EXTERNAL_USB/Features_Phrampus_flipped")
restrepo_files = []

for (dirpath, dirnames, filenames) in walk(restrepo_features_path):
    restrepo_files.extend(filenames)

restrepo_files.sort()
len(restrepo_files)

rootgrp = xr.open_dataset(Path(restrepo_features_path / restrepo_files[0]), format = 'NETCDF4')
latitudes = rootgrp.variables['lat']
longitudes = rootgrp.variables['lon']

restrepo_features = np.empty([len(df),len(restrepo_files[0:len(restrepo_files)])])
restrepo_features[:] = np.nan
restrepo_X_mean = np.empty(0)
restrepo_X_std = np.empty(0)

file_idx = -1
corrupted_files = 0


In [ ]:
def find_nearest(value,vector):
    vector = np.float64(vector)-np.float64(value);
    return np.argmin(np.abs(vector))

for file in restrepo_files[0:len(restrepo_files)]:
    if file == "GL_ELEVATION_M_ASL_ETOPO2v2.5m.grd":
        
        try: 
            rootgrp = xr.open_dataset(Path(restrepo_features_path / file), format = 'NETCDF4')
        except:
            restrepo_X_mean = np.append(restrepo_X_mean,np.nan)
            restrepo_X_std = np.append(restrepo_X_std,np.nan)
            corrupted_files = corrupted_files + 1
            continue
    
        
    #     latitudes = rootgrp.variables['lats'] # in theory all the same
    #     longitudes = rootgrp.variables['lons']
        
        dataset = rootgrp.variables['data']
    
        restrepo_X_mean = np.append(restrepo_X_mean,np.nanmean(dataset))
        restrepo_X_std = np.append(restrepo_X_std,np.nanstd(dataset))
        #plt.figure(figsize=([20,10]), dpi=300);plt.imshow(np.rot90(np.rot90(np.fliplr(dataset_numpy))))
        dataset_numpy = dataset.to_numpy()
        #dataset_numpy = np.rot90(np.rot90(np.fliplr(dataset_numpy)))
        # ds = xr.Dataset(
    
        #     data_vars=dict(
    
        #         data=(['lat', 'lon'],dataset_numpy)
    
        #     ),
    
        #     coords=dict(
    
        #         lon=( longitudes),
    
        #         lat=(latitudes)
        #     ),
        #     attrs=dict(description="feature data set for sediment accumulation prediction"),    
        # )
        # ds.to_netcdf(path = "/media/nparameswaran/EXTERNAL_USB/Features_Phrampus_flipped/"+file)
        # plt.figure(figsize=([20,10]), dpi=300)
        # plt.title(file[:-3])
        # plt.imshow(dataset)
        # plt.colorbar()
        # if file[:-3]!='.':
        #     plt.savefig( feature_image_path +file[:-4]+".png")
        # else:
        #     plt.savefig( feature_image_path +file[:-3]+".png")
        # plt.close()
        
        # latitudes_labels=[]
        # longitudes_labels=[]
        filename=file
    else:
        continue

In [ ]:
plt.imshow(dataset_numpy)

In [ ]:
#mask1 is <-200m
#mask2 is >-200m
a = dataset_numpy > -200
b = dataset_numpy < 0
mask1 = np.where(np.logical_and(a, b), 1, np.nan) #255 and 0 are the binary values for true and false
mask2 = np.where(dataset_numpy <= -200 , 1,np.nan) 
#mask1 = np.rot90(np.rot90(np.fliplr(mask1)))
df1 = pd.DataFrame()
df2 = pd.DataFrame()

lat_idx_shallow = []
lon_idx_shallow = []
elevation_cs = []


dataset = mask1

for data_idx in range(0,len(df)):
    lat_idx = find_nearest(df.iloc[data_idx,0], latitudes)
    lon_idx = find_nearest(df.iloc[data_idx,1], longitudes)
    #plt.figure(data_idx)
    #plt.imshow(mask1[-10:10,-10:10])
    if mask1[lat_idx, lon_idx]==1:
        elevation_cs.append(dataset_numpy[lat_idx, lon_idx])
        #plt.imshow(mask1[-10:10,-10:10])
        if df1.empty:
            df1 = df.iloc[data_idx]
        else:
            df1 = pd.concat([df1, df.iloc[data_idx]], axis=1, ignore_index=True)
    if mask2[lat_idx, lon_idx]==1:
        if df2.empty:
            df2 = df.iloc[data_idx]
        else:
            df2 = pd.concat([df2, df.iloc[data_idx]], axis=1, ignore_index=True)
    if data_idx%1000 == 0:
        print(data_idx)
    # latitudes_labels.append(latitudes[lat_idx])
    # longitudes_labels.append(longitudes[lon_idx])

df1 = df1.transpose()
df2 = df2.transpose()

#df1 = df1[df1.iloc[:,2] < 5000 ]
#df2 = df2[df2.iloc[:,2] < 1000 ]

np.save(str(project_root / "data" / "interim" / "masks"),mask2)

df1 = df1[df1["Longitude"]>-179.5]
df1 = df1[df1["Longitude"]<179.5]
df1 = df1[df1["Latitude"]>-89.5]
df1 = df1[df1["Latitude"]<89.5]

df2 = df2[df2["Longitude"]>-179.5]
df2 = df2[df2["Longitude"]<179.5]
df2 = df2[df2["Latitude"]>-89.5]
df2 = df2[df2["Latitude"]<89.5]


df1.to_csv(str(Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/labels/sedrate_continentalshelves.csv")) 
df2.to_csv(str(Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/labels/sedrate_deep.csv"))

#df1 = df1[df1["Sedimentation rate [cm/ka]"]<1]

# Load world map
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# # Create GeoDataFrame 1
# geometry1 = [Point(xy) for xy in zip(df1['Longitude'], df1['Latitude'])]
# gdf1 = GeoDataFrame(df1, geometry=geometry1)

# # Create GeoDataFrame 2
# geometry2 = [Point(xy) for xy in zip(df2['Longitude'], df2['Latitude'])]
# gdf2 = GeoDataFrame(df2, geometry=geometry2)

# # Plot GeoDataFrame 1 on world map
# ax = world.plot(figsize = (20, 12))
# gdf1.plot(ax=ax, column='Sedimentation rate [cm/ka]',  color='blue', legend = "Sedimentation rates in locations with elevation>-200m")

# # Plot GeoDataFrame 2 on the same world map
# gdf2.plot(ax=ax, column='Sedimentation rate [cm/ka]',  color='red', legend = "Sedimentation rates in locations with elevation<-200m")


# Show legend
# ax.legend()

# Show the plot
"""
plt.figure(figsize = ([20,10]))#, dpi = 200)
#plt.title("Continental shelves(elevation of 0  to -200 $m$)")

plt.imshow(np.rot90(np.rot90(np.fliplr(mask1))))
# plt.show()

plt.figure(figsize = ([20,10]))#, dpi = 200)
#plt.title("Continental shelves(elevation of 0  to -200 $m$)")

# Create a custom colormap for True and False values
cmap = plt.cm.get_cmap('viridis', 2)  # Use 'viridis' colormap with 2 colors (for True and False)

# Plot the mask2 array
plt.imshow(np.rot90(np.rot90(np.fliplr(mask2))), cmap=cmap)

# Set the ticks and labels for the colorbar
cbar = plt.colorbar(ticks=[0.25, 0.75])
cbar.ax.set_yticklabels(['False', 'True'])  # Replace the default ticks (0 and 1) with 'False' and 'True'

# Set the colorbar label
cbar.set_label('Mask Value', rotation=270, labelpad=20)

# Show the plot
plt.show()
"""

In [ ]:
# Load the CSV files and combine into a single DataFrame
df1 = pd.read_csv(str(Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/labels/sedrate_continentalshelves.csv"))
df2 = pd.read_csv(str(Path().resolve().parents[2] / "nn4sedratetoc" / "data/raw/labels/sedrate_deep.csv"))
df = pd.concat([df1, df2], ignore_index=True)


In [ ]:
df

In [ ]:
df1["Sedimentation rate [cm/ka]"] = pd.to_numeric(df1["Sedimentation rate [cm/ka]"])
df2["Sedimentation rate [cm/ka]"] = pd.to_numeric(df2["Sedimentation rate [cm/ka]"])

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cmcrameri.cm as cmc
import cmocean as cmo

# Assuming 'world' is a GeoDataFrame of the world boundaries
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Assuming 'df1' is your DataFrame with Latitude, Longitude, and TOC columns
df1 = df1.drop_duplicates()

# Create a GeoDataFrame
geometry = [Point(xy) for xy in zip(df1['Longitude'], df1['Latitude'])]
gdf = GeoDataFrame(df1, geometry=geometry)

# Plotting
fig, ax = plt.subplots(figsize=(20, 10), dpi=300)

# Plotting the world map with a light blue background
ax.set_facecolor('black')  # Light blue color

# Plotting the GeoDataFrame with log-scale color map and smaller points
#gdf.plot(ax=ax, column=df1["Sedimentation rate [cm/ka]"], cmap='cubehelix', norm=LogNorm(vmin=1e-1, vmax = 15), legend=True, markersize=3, legend_kwds={
#        "shrink":.75, "label": "Sedimentation rate [cm/ka]", "pad" : 0.05})
cax = gdf.plot(ax=ax, column=df1["Sedimentation rate [cm/ka]"], cmap=cmc.hawaii, norm=LogNorm(vmin=1e-1, vmax=5000), legend=True, markersize=3
               , legend_kwds={
    "shrink": 0.75,
    "label": "Sedimentation rate [cm/ka]",
    "pad": 0.01# Title of the color bar
})


world.plot(ax=ax, alpha=0.7, color='grey')


# Set font size for all elements
plt.xlabel("Longitude", fontsize=20)
plt.ylabel("Latitude", fontsize=20)
#plt.title("TOC data points", fontsize=20)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
#plt.legend(fontsize=20)
plt.xlim(-180, 180)
plt.ylim(-90, 90)
#plt.title("TOC data points")
plt.tight_layout()
plt.show()



In [ ]:
df2["Sedimentation rate [cm/ka]"]

In [ ]:
df1

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cmcrameri.cm as cmc
import cmocean as cmo

# Assuming 'world' is a GeoDataFrame of the world boundaries
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# Assuming 'df1' is your DataFrame with Latitude, Longitude, and TOC columns
df1 = df1.drop_duplicates()

# Create a GeoDataFrame
geometry = [Point(xy) for xy in zip(df2['Longitude'], df2['Latitude'])]
gdf = GeoDataFrame(df2, geometry=geometry)

# Plottingdf1
fig, ax = plt.subplots(figsize=(20, 10), dpi=300)

# Plotting the world map with a light blue background
ax.set_facecolor('black')  # Light blue color

# Plotting the GeoDataFrame with log-scale color map and smaller points
#gdf.plot(ax=ax, column=df1["Sedimentation rate [cm/ka]"], cmap='cubehelix', norm=LogNorm(vmin=1e-1, vmax = 15), legend=True, markersize=3, legend_kwds={
#        "shrink":.75, "label": "Sedimentation rate [cm/ka]", "pad" : 0.05})
cax = gdf.plot(ax=ax, column=df2["Sedimentation rate [cm/ka]"], cmap=cmc.hawaii, norm=LogNorm(vmin=1e-1, vmax=5000), legend=True, markersize=3
               , legend_kwds={
    "shrink": 0.75,
    "label": "Sedimentation rate [cm/ka]",
    "pad": 0.01# Title of the color bar
})


world.plot(ax=ax, alpha=0.7, color='grey')


# Set font size for all elements
plt.xlabel("Longitude", fontsize=20)
plt.ylabel("Latitude", fontsize=20)
#plt.title("TOC data points", fontsize=20)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
#plt.legend(fontsize=20)
plt.xlim(-180, 180)
plt.ylim(-90, 90)
#plt.title("TOC data points")
plt.tight_layout()
plt.show()



In [ ]:
df2["Latitude"].max()

In [ ]:
# Only plot the global distributions (world maps) for df1 (sedimentation rate) and df2 (mass accumulation rate), binned and colored

# Remove NaN and inf values for plotting
df1_clean = df1.dropna(subset=['Sedimentation rate [cm/ka]'])
df1_clean = df1_clean[np.isfinite(df1_clean['Sedimentation rate [cm/ka]'])]

df2_clean = df2.dropna(subset=['MAR(g/cm**2/ka)'])
df2_clean = df2_clean[np.isfinite(df2_clean['MAR(g/cm**2/ka)'])]

# Define bins and colors
bins_sr = [0, 0.1, 1, 10, 100, 1000, 10000]
labels_sr = ['0-0.1', '0.1-1', '1-10', '10-100', '100-1000', '1000+']
colors = ['darkblue', 'blue', 'cyan', 'green', 'orange', 'red']
from matplotlib.colors import ListedColormap
cmap = ListedColormap(colors)

# Bin the sedimentation rate for df1
df1_clean['sr_bin'] = pd.cut(df1_clean['Sedimentation rate [cm/ka]'], bins=bins_sr, labels=labels_sr, include_lowest=True)
# Bin the MAR for df2
df2_clean['mar_bin'] = pd.cut(df2_clean['MAR(g/cm**2/ka)'], bins=bins_sr, labels=labels_sr, include_lowest=True)

# Plot 1: Global map for df1 (sedimentation rate, binned)
fig, ax = plt.subplots(figsize=(20, 10), dpi=300)
ax.set_facecolor('black')
geometry1 = [Point(xy) for xy in zip(df1_clean['Longitude'], df1_clean['Latitude'])]
gdf1 = GeoDataFrame(df1_clean, geometry=geometry1)
gdf1.plot(ax=ax, column='sr_bin', cmap=cmap, markersize=3, legend=True)
world.plot(ax=ax, alpha=0.7, color='grey')
ax.set_title('Global Distribution of Sedimentation Rate (Binned) - DF1', fontsize=18)
ax.set_xlabel('Longitude', fontsize=16)
ax.set_ylabel('Latitude', fontsize=16)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
plt.tight_layout()
plt.show()

# Plot 2: Global map for df2 (mass accumulation rate, binned)
fig, ax = plt.subplots(figsize=(20, 10), dpi=300)
ax.set_facecolor('black')
geometry2 = [Point(xy) for xy in zip(df2_clean['Longitude'], df2_clean['Latitude'])]
gdf2 = GeoDataFrame(df2_clean, geometry=geometry2)
gdf2.plot(ax=ax, column='mar_bin', cmap=cmap, markersize=3, legend=True)
world.plot(ax=ax, alpha=0.7, color='grey')
ax.set_title('Global Distribution of Mass Accumulation Rate (Binned) - DF2', fontsize=18)
ax.set_xlabel('Longitude', fontsize=16)
ax.set_ylabel('Latitude', fontsize=16)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Flatten the dataset_numpy to 1D array of elevations
elevations = dataset_numpy.flatten()

# Remove NaN and inf values
elevations = elevations[np.isfinite(elevations)]

# Filter to only elevations less than 5 meters
elevations_sub5 = elevations[elevations < 5]

# Define fine bins for this narrow range
bins = np.arange(elevations_sub5.min(), 5.1, 0.5)

# Plot histogram (frequency plot) for elevations < 5m with log scale on y-axis
plt.figure(figsize=(10, 6))
plt.hist(elevations_sub5, bins=bins, color='royalblue', edgecolor='black', alpha=0.7)
plt.xlabel('Elevation (m)', fontsize=14, fontweight='bold')
plt.ylabel('Number of grid cells', fontsize=14, fontweight='bold')
plt.title('Frequency Distribution of Elevation < 5 m in Bathymetry Dataset', fontsize=16, fontweight='bold', pad=20)
plt.yscale('log')  # Set y-axis to log scale

# Add reference lines for important bathymetric boundaries in this range
plt.axvline(x=0, color='green', linestyle='--', linewidth=2, label='Sea Level (0 m)')

plt.legend(frameon=True, fontsize=12)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Print some statistics for elevations < 5m
print("=== ELEVATION (<5m) DISTRIBUTION STATISTICS ===")
print(f"Total valid grid cells: {len(elevations_sub5):,}")
print(f"Elevation range: {elevations_sub5.min():.2f} m to {elevations_sub5.max():.2f} m")
print(f"Median elevation: {np.median(elevations_sub5):.2f} m")
print(f"Mean elevation: {np.mean(elevations_sub5):.2f} m")
print(f"Standard deviation: {np.std(elevations_sub5):.2f} m")
print(f"25th percentile: {np.percentile(elevations_sub5, 25):.2f} m")
print(f"75th percentile: {np.percentile(elevations_sub5, 75):.2f} m")

In [ ]:
# Add an 'elevation' column to df by extracting elevation from dataset_numpy at each (lat, lon) point
# and plot the points on top of the elevation map

import matplotlib.pyplot as plt
import numpy as np

# Define lats and lons for the grid based on dataset_numpy shape and global coverage
nlat, nlon = dataset_numpy.shape  # Should be (2160, 4320)
lats = np.linspace(-90, 90, nlat)
lons = np.linspace(-180, 180, nlon)

def get_elevation_for_points(df, lats, lons, dataset_numpy):
    """
    For each row in df, find the closest grid cell in (lats, lons) and get the elevation from dataset_numpy.
    """
    lats = np.asarray(lats)
    lons = np.asarray(lons)
    elevations = []
    for idx, row in df.iterrows():
        lat = row['Latitude']
        lon = row['Longitude']
        # Find the nearest index in lats and lons
        lat_idx = np.abs(lats - lat).argmin()
        lon_idx = np.abs(lons - lon).argmin()
        # Get elevation
        elevation = dataset_numpy[lat_idx, lon_idx]
        elevations.append(elevation)
    return elevations

# Add the elevation column to df
df['elevation'] = get_elevation_for_points(df, lats, lons, dataset_numpy)

# Plot the elevation map and overlay the points (make points smaller)
plt.figure(figsize=(12, 6))
extent = [lons.min(), lons.max(), lats.min(), lats.max()]
plt.imshow(dataset_numpy, extent=extent, origin='lower', aspect='auto', cmap='terrain')
plt.colorbar(label='Elevation (m)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Elevation Map with Data Points Overlayed')

# Overlay the points from df with smaller size
plt.scatter(df['Longitude'], df['Latitude'], c='red', s=8, label='Data Points', edgecolor='black')
plt.legend()
plt.tight_layout()
plt.show()

# Show the first few rows of df with the new elevation column
print(df[['Latitude', 'Longitude', 'elevation']].head())

# Overlay the frequency distributions of (1) all grid cells <0m and (2) measurement points <0m
# Use thicker bins and add a KDE plot for the measurement points

import seaborn as sns

plt.figure(figsize=(6, 4), dpi=300)

# 1. All grid cells: elevations < 0m
elevations = dataset_numpy.flatten()
elevations = elevations[np.isfinite(elevations)]
elevations_sub0 = elevations[elevations < 0]

# 2. Measurement points: elevations < 0m
elevs = df['elevation'].dropna().values
elevs_sub0 = elevs[elevs < 0]

# Define thicker bins (e.g., 10 m width)
min_bin = np.floor(min(elevations_sub0.min(), elevs_sub0.min()) / 10) * 10
max_bin = 0
bins = np.arange(min_bin, max_bin + 10, 10)

# Plot all grid cells histogram (background, semi-transparent)
plt.hist(elevations_sub0, bins=bins, color='gray',  alpha=0.4, label='All Grid Cells (<0m)')

# Plot measurement points histogram (foreground)
plt.hist(elevs_sub0, bins=bins, color='royalblue', alpha=0.7, label='Measurement Points (<0m)')

# KDE plot for measurement points
if len(elevs_sub0) > 1:
    sns.kdeplot(elevs_sub0, color='crimson', linewidth=2, label='Measurement KDE', bw_adjust=1.2, clip=(min_bin, 0))

plt.xlabel('Elevation (m)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Grid Cells / Measurements', fontsize=12, fontweight='bold')
plt.title('Frequency Distribution of Elevation < 0 m', fontsize=13, fontweight='bold', pad=15)
plt.yscale('log')
plt.ylim(1, 100000)
plt.axvline(x=0, color='green', linestyle='--', linewidth=2, label='Sea Level (0 m)')
plt.axvline(x=-200, color='orange', linestyle='--', linewidth=2, label='200 m Bathymetry')
plt.legend(frameon=True, fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()


In [ ]:
# Add a 'porosity_graw_derived' column to df by extracting porosity from porosity_graw at each (lat, lon) point
# and plot the points on top of the porosity map

import matplotlib.pyplot as plt
import numpy as np
porosity_graw_masked_rotated = np.rot90(np.rot90(np.fliplr(porosity_graw_masked)))
# Define lats and lons for the grid based on porosity_graw shape and global coverage
nlat, nlon = porosity_graw_masked_rotated.shape  # Should be (2160, 4320) or similar
lats = np.linspace(-90, 90, nlat)
lons = np.linspace(-180, 180, nlon)

def get_porosity_for_points(df, lats, lons, porosity_map):
    """
    For each row in df, find the closest grid cell in (lats, lons) and get the porosity from porosity_map.
    """
    lats = np.asarray(lats)
    lons = np.asarray(lons)
    porosities = []
    for idx, row in df.iterrows():
        lat = row['Latitude']
        lon = row['Longitude']
        # Find the nearest index in lats and lons
        lat_idx = np.abs(lats - lat).argmin()
        lon_idx = np.abs(lons - lon).argmin()
        # Get porosity
        porosity = porosity_map[lat_idx, lon_idx]
        porosities.append(porosity)
    return porosities

# Add the porosity_graw_derived column to df
df['porosity_graw_derived'] = get_porosity_for_points(df, lats, lons, porosity_graw_masked_rotated)

# Plot the porosity map and overlay the points (make points smaller)
plt.figure(figsize=(12, 6))
extent = [lons.min(), lons.max(), lats.min(), lats.max()]
plt.imshow(porosity_graw_masked_rotated, extent=extent, origin='lower', aspect='auto', cmap='cubehelix', vmin=40, vmax=100)
plt.colorbar(label='Porosity (Graw-derived, %)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Graw-derived Porosity Map with Data Points Overlayed')

# Overlay the points from df with smaller size
plt.scatter(df['Longitude'], df['Latitude'], c='red', s=8, label='Data Points', edgecolor='black')
plt.legend()
plt.tight_layout()
plt.show()

# Show the first few rows of df with the new porosity_graw_derived column
print(df[['Latitude', 'Longitude', 'porosity_graw_derived']].head())

# Overlay the frequency distributions of (1) all grid cells porosity and (2) measurement points porosity
# Use thicker bins and add a KDE plot for the measurement points

import seaborn as sns

plt.figure(figsize=(6, 4), dpi=300)

# 1. All grid cells: porosity (valid only)
porosity_all = porosity_graw_masked.flatten()
porosity_all = porosity_all[np.isfinite(porosity_all)]

# 2. Measurement points: porosity
porosity_pts = df['porosity_graw_derived'].dropna().values

import numpy as np

# Convert 'masked' to np.nan, and cast to float (int not possible with nan)
porosity_pts_clean = np.array([
    float(x) if x != 'masked' else np.nan
    for x in porosity_pts
])

# If you want integer values (will be float if any nan present)
# porosity_pts_int = porosity_pts_clean.astype(float)  # Already float, but for clarity

porosity_pts_clean

# Define bins (e.g., 2% width)
min_bin = np.floor(min(porosity_all.min(), porosity_pts.min()) / 2) * 2
max_bin = np.ceil(max(porosity_all.max(), porosity_pts.max()) / 2) * 2
bins = np.arange(min_bin, max_bin + 2, 2)

# Plot all grid cells histogram (background, semi-transparent)
plt.hist(porosity_all, bins=bins, color='gray',  alpha=0.4, label='All Grid Cells')

# Plot measurement points histogram (foreground)
plt.hist(porosity_pts_clean, bins=bins, color='royalblue', alpha=0.7, label='Measurement Points')

# KDE plot for measurement points
if len(porosity_pts_clean) > 1:
    sns.kdeplot(porosity_pts_clean, color='crimson', linewidth=2, label='Measurement KDE', bw_adjust=1.2, clip=(min_bin, max_bin))

plt.xlabel('Porosity (%)', fontsize=12, fontweight='bold')
plt.ylabel('Number of Grid Cells / Measurements', fontsize=12, fontweight='bold')
plt.title('Frequency Distribution of Graw-derived Porosity', fontsize=13, fontweight='bold', pad=15)
plt.yscale('log')
plt.ylim(1, 10000000)
plt.axvline(x=55, color='orange', linestyle='--', linewidth=2, label='Porosity = 55%')
plt.legend(frameon=True, fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()


In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt

# Generate grid points
num_lat, num_lon = 2160, 4320
lat_grid = np.linspace(-90, 90, num_lat)
lon_grid = np.linspace(-180, 180, num_lon)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

# Flatten for scatter plot
lat_flat = lat_mesh.ravel()
lon_flat = lon_mesh.ravel()

# Remove red points on land using land_map
# land_map shape: (2160, 4320), 1=land, 0=sea

land_map = np.rot90(np.rot90(np.fliplr(land_map)))
# Flatten land_map to match lat/lon_flat
land_flat = land_map.ravel()
sea_mask = (land_flat == 0)  # True for sea, False for land
"""
plt.figure(figsize=(8, 6))

# Plot only sea grid points as small red dots
plt.scatter(lon_flat[sea_mask], lat_flat[sea_mask], color='red', s=0.01, alpha=0.5, label='2160x4320 Grid (Sea Only)')

# Overlay the df points
plt.scatter(df['Longitude'], df['Latitude'], c='royalblue', edgecolor='k', alpha=0.7, label='df points')

plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title('Scatter plot of lat/lon from df over 2160x4320 Grid', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()
"""

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# --- Prepare points as before ---

# Load land mask
land_file = "../../data/raw/island_map.npy"
land_map = np.load(land_file)
land_map[np.isnan(land_map)] = 1

# Generate grid points
num_lat, num_lon = 2160, 4320
lat_grid = np.linspace(-90, 90, num_lat)
lon_grid = np.linspace(-180, 180, num_lon)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

# Flatten for processing
lat_flat = lat_mesh.ravel()
lon_flat = lon_mesh.ravel()

# Remove red points on land using land_map
# land_map shape: (2160, 4320), 1=land, 0=sea
land_map = np.rot90(np.rot90(np.fliplr(land_map)))
land_flat = land_map.ravel()
sea_mask = (land_flat == 0)  # True for sea, False for land

# Red points: sea grid points
red_lats = lat_flat[sea_mask]
red_lons = lon_flat[sea_mask]
red_points = np.column_stack([red_lats, red_lons])

# Blue points: df points
blue_lats = df['Latitude'].values
blue_lons = df['Longitude'].values
blue_points = np.column_stack([blue_lats, blue_lons])

# Helper: haversine distance in km (vectorized for cdist)
def haversine_vec(latlon1, latlon2):
    # latlon1, latlon2: (..., 2) arrays, lat/lon in degrees
    R = 6371.0
    lat1 = np.radians(latlon1[..., 0])
    lon1 = np.radians(latlon1[..., 1])
    lat2 = np.radians(latlon2[..., 0])
    lon2 = np.radians(latlon2[..., 1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# --- Nearest neighbour distributions (like @file_context_0) ---

# For memory, sample a subset of red and blue points
max_red = 2000
max_blue = 2000
rng = np.random.default_rng(42)
if len(red_points) > max_red:
    red_idx = rng.choice(len(red_points), max_red, replace=False)
    red_sample = red_points[red_idx]
else:
    red_sample = red_points
if len(blue_points) > max_blue:
    blue_idx = rng.choice(len(blue_points), max_blue, replace=False)
    blue_sample = blue_points[blue_idx]
else:
    blue_sample = blue_points

# 1. Nearest neighbour blue-to-blue (excluding self)
blue_dmat = cdist(blue_sample, blue_sample, lambda u, v: haversine_vec(u, v))
np.fill_diagonal(blue_dmat, np.inf)
blue_nn = np.min(blue_dmat, axis=1)

# 2. Nearest neighbour red-to-red (excluding self)
red_dmat = cdist(red_sample, red_sample, lambda u, v: haversine_vec(u, v))
np.fill_diagonal(red_dmat, np.inf)
red_nn = np.min(red_dmat, axis=1)

# 3. Nearest neighbour blue-to-red (for each blue, min distance to any red)
blue_to_red_dmat = cdist(blue_sample, red_sample, lambda u, v: haversine_vec(u, v))
blue_to_red_nn = np.min(blue_to_red_dmat, axis=1)

# 4. Nearest neighbour red-to-blue (for each red, min distance to any blue)
red_to_blue_dmat = cdist(red_sample, blue_sample, lambda u, v: haversine_vec(u, v))
red_to_blue_nn = np.min(red_to_blue_dmat, axis=1)

# --- Plot neighbourhood distributions ---

plt.figure(figsize=(12, 6))
plt.hist(red_nn, bins=100, alpha=0.5, label='Red-Red (sea grid) NN', density=True, color='red')
plt.hist(blue_nn, bins=100, alpha=0.5, label='Blue-Blue (df) NN', density=True, color='blue')
plt.hist(blue_to_red_nn, bins=100, alpha=0.5, label='Blue-to-Red NN', density=True, color='purple')
plt.hist(red_to_blue_nn, bins=100, alpha=0.5, label='Red-to-Blue NN', density=True, color='orange')
plt.xlabel('Nearest Neighbour Distance (km)')
plt.ylabel('Density')
plt.title('Nearest Neighbour Distance Distributions')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# --- Remove negative values for KDE plotting (cut at 0) ---
blue_nn_pos = blue_nn[blue_nn >= 0] + 1e-3  # add small constant as before
red_to_blue_nn_pos = red_to_blue_nn[red_to_blue_nn >= 0] + 1e-3

# --- Figure & style ---
plt.style.use('default')  # Use matplotlib's default style
fig, ax = plt.subplots(figsize=(3, 1.5), dpi=300)
print(np.min(blue_nn_pos), np.max(blue_nn_pos))
# --- KDEs ---
sns.kdeplot(
    blue_nn_pos, label='Sample→Sample', color='navy',
    fill=True, lw=0.8, alpha=0.35, ax=ax, bw_adjust=0.9, common_norm=False, clip=(0, None)
)
sns.kdeplot(
    red_to_blue_nn_pos, label='Prediction→Sample', color='darkorange',
    fill=True, lw=0.8, alpha=0.35, ax=ax, bw_adjust=0.9, common_norm=False, clip=(0, None)
)

# --- Medians ---
m_blue = np.median(blue_nn)
m_rb   = np.median(red_to_blue_nn)
ax.axvline(m_blue, color='navy', ls='--', lw=0.8, alpha=0.7)
ax.axvline(m_rb,   color='darkorange', ls='--', lw=0.8, alpha=0.7)

# --- Axes formatting ---
ax.set_xlabel('Nearest neighbor distance (km)', fontsize=8)
ax.set_ylabel('Density', fontsize=8)
ax.set_xlim(left=0)  # Cut distribution at zero
ax.set_ylim(0, 0.011)
ax.set_xlim(-100,3500)
ax.tick_params(axis='both', which='major', labelsize=7)
ax.xaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f'))
ax.yaxis.set_major_locator(mtick.MaxNLocator(nbins=5))
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.3f'))

# --- Annotate medians ---
ax.text(m_blue+200, ax.get_ylim()[1]*0.9, f"{m_blue:.0f}", 
        color='navy', fontsize=7, ha='right', va='bottom', rotation=0)
ax.text(m_rb+400, ax.get_ylim()[1]*0.9, f"{m_rb:.0f}", 
        color='darkorange', fontsize=7, ha='right', va='bottom', rotation=0)

# --- Legend & grid ---
ax.legend(frameon=False, fontsize=7, loc='upper right')
ax.grid(True, which='major', linestyle='-', linewidth=0.5, alpha=0.5)  # Use matplotlib style grid

plt.tight_layout(pad=0.2)
plt.show()